<a href="https://colab.research.google.com/github/Protein-Function-Prediction/COMP3608ProteinFunctionPrediction/blob/protein-function-prediction-changes/Protein_function_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Function Prediction – Data Preparation Pipeline
**Project:** COMP 3608 B‑rank mission  
**This notebook:** loads datasets from uploaded zip files → cleans & processes each dataset → engineers features → saves model-ready arrays to `data/processed/`.

**Datasets:**
- **Dataset 1** – `Bioinformatics_Dataset.zip` → Simulated protein dataset with physicochemical features (Portuguese columns, 5-class classification)
- **Dataset 2** – `Protein_sequences_w_GO_Annotations.zip` → Real UniProt proteins with full feature set + GO annotations
- **Dataset 3** – `Human_Protein_Sequences_and_Function_Annotations.zip` → Human proteome FASTA sequences + GO annotation text file

In [51]:
# 0. Environment
import warnings
import zipfile
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Create directory structure
RAW_DIR  = Path('data/raw')
PROC_DIR = Path('data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Standard amino-acid alphabet used throughout
AA = list('ACDEFGHIKLMNPQRSTVWY')

print('✔ Environment ready')
print(f'  RAW_DIR  : {RAW_DIR}')
print(f'  PROC_DIR : {PROC_DIR}')

✔ Environment ready
  RAW_DIR  : data/raw
  PROC_DIR : data/processed


In [2]:
import kagglehub

# GO annotations (real UniProt proteins)
path_go = kagglehub.dataset_download(
    "nikitamanaenkov/protein-sequences-with-go-annotations",
    output_dir=str(RAW_DIR / 'go_annotations')
)
# Simulated bioinformatics dataset 1
path_sim1 = kagglehub.dataset_download(
    "willianoliveiragibin/bioinformatics-simulated",
    output_dir=str(RAW_DIR / 'simulated_1')
)
# Simulated bioinformatics dataset 2
path_human = kagglehub.dataset_download(
    "usamaraheem/human-protein-sequences-and-function-annotations",
    output_dir=str(RAW_DIR / 'human_protein')
)
print("✔ All datasets downloaded to:", str(RAW_DIR))

100%|██████████| 32.8M/32.8M [00:00<00:00, 40.8MB/s]

Extracting files...


100%|██████████| 2.52M/2.52M [00:00<00:00, 143MB/s]

Extracting files...


100%|██████████| 16.5M/16.5M [00:00<00:00, 39.7MB/s]

Extracting files...


✔ All datasets downloaded to: data/raw


---
# Dataset 1 : Bioinformatics Simulated

In [7]:
# 1.1  Load
# The kagglehub.dataset_download function extracts the contents directly into the specified directory.
# So, we should read the CSV directly from the directory, not try to open a zip file.
df1 = pd.read_csv(RAW_DIR / 'simulated_1' / 'proteinas_train new.csv')

print(f'\nShape : {df1.shape}')
df1.head()


Shape : (16000, 10)


,ID_Proteína,Sequência,Massa_Molecular,Ponto_Isoelétrico,Hidrofobicidade,Carga_Total,Proporção_Polar,Proporção_Apolar,Comprimento_Sequência,Classe
0,TRAIN_P00001,GNMRFVLHDEETHWGTLRTTLNCVPSDIYTISGEDSLFWGMAHPFC...,20.362.946.799.999.900,48.661.226.272.583,14.942.528.735.632.100,-3,2.413.793.103.448.270,40.804.597.701.149.400,174,Estrutural
1,TRAIN_P00002,LFKMQCSFYLLYLAKEAASYQVSMNMLCYEWYNYVYQVTVILRLSR...,9.328.790.899.999.990,6.298.635.673.522.940,21.710.526.315.789.400,0,21.052.631.578.947.300,5.131.578.947.368.420,76,Estrutural
2,TRAIN_P00003,PAHLWPYWRFYVWIVFYGYHNPNYHFGMKEVKERPDCKNCTVAVLF...,176.163.852,845.897.731.781.006,19.256.756.756.756.700,8,14.189.189.189.189.100,46.621.621.621.621.600,148,Estrutural
3,TRAIN_P00004,GEAFSRPHCFACAATKKGFPWARMCCTTSMAMDGVQSKMHKSKHRF...,3.524.429.680.000.000,8.448.340.034.484.860,16.047.297.297.297.200,21,1.891.891.891.891.890,40.878.378.378.378.300,296,Estrutural
4,TRAIN_P00005,HYVFQGLMLHCGGYMITACGFGVIFPEQMTREGLIMHTARAHHFLI...,3.455.799.310.000.000,769.630.641.937.256,1.404.109.589.041.090,18,20.205.479.452.054.700,3.801.369.863.013.690,292,Receptora


Translating the columns and class labels from Portuguese to English for better readability.

In [8]:
# ── 1.2  Rename columns (Portuguese → English) ─────────────────────────────
df1 = df1.rename(columns={
    'ID_Proteína'         : 'Protein_ID',
    'Sequência'           : 'Sequence',
    'Massa_Molecular'     : 'Molecular_Weight',
    'Ponto_Isoelétrico'   : 'Isoelectric_Point',
    'Hidrofobicidade'     : 'Hydrophobicity',
    'Carga_Total'         : 'Net_Charge',
    'Proporção_Polar'     : 'Polar_Ratio',
    'Proporção_Apolar'    : 'NonPolar_Ratio',
    'Comprimento_Sequência': 'Sequence_Length',
    'Classe'              : 'Class'
})

# ── 1.3  Translate class labels ────────────────────────────────────────────
df1['Class'] = df1['Class'].replace({
    'Estrutural': 'Structural',
    'Receptora' : 'Receptor',
    'Enzima'    : 'Enzyme',
    'Transporte': 'Transporter',
    'Outras'    : 'Others',
})

print('Class distribution after translation:')
print(df1['Class'].value_counts())

Class distribution after translation:
Class
Enzyme         3235
Structural     3232
Transporter    3225
Others         3183
Receptor       3125
Name: count, dtype: int64


In [9]:
# ── 1.4  Quality checks ────────────────────────────────────────────────────
print('Null values per column:')
print(df1.isnull().sum())
print()

# Drop rows with null sequences or missing physicochemical values
df1 = df1.dropna(subset=['Sequence', 'Molecular_Weight', 'Isoelectric_Point',
                          'Hydrophobicity', 'Net_Charge', 'Polar_Ratio',
                          'NonPolar_Ratio', 'Sequence_Length', 'Class'])

# Remove sequences containing non-standard amino acid characters
valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
mask = df1['Sequence'].apply(lambda s: set(str(s).upper()).issubset(valid_aa))
removed = (~mask).sum()
df1 = df1[mask].reset_index(drop=True)
print(f'Rows removed (invalid AA characters): {removed}')
print(f'Remaining rows: {len(df1)}')

Null values per column:
Protein_ID           0
Sequence             0
Molecular_Weight     0
Isoelectric_Point    0
Hydrophobicity       0
Net_Charge           0
Polar_Ratio          0
NonPolar_Ratio       0
Sequence_Length      0
Class                0
dtype: int64

Rows removed (invalid AA characters): 0
Remaining rows: 16000


---
# Dataset 2 : Protein Sequences with GO Annotations

In [41]:
# ── 2.1  Load ──────────────────────────────────────────────────────────────
# The kagglehub.dataset_download function extracts the contents directly into the specified directory.
# So, we should read the CSV directly from the directory, not try to open a zip file.
df2_raw = pd.read_csv(RAW_DIR / 'go_annotations' / 'proteins.csv')

print(f'\nShape : {df2_raw.shape}')
df2_raw.head(3)


Shape : (81000, 37)


,Entry,Sequence,GO_list,GO,GO_id,GO_namespace,GO_parents,GO_children,seq_length,mol_weight,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,C0SPC1,MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...,"['cytoplasm [GO:0005737]', 'ATP binding [GO:00...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",269.0,30790.1862,...,12.0,11.0,11.0,10.0,9.0,14.0,8.0,14.0,10.0,8.0
1,O05512,MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...,"['extracellular region [GO:0005576]', 'mannan ...",extracellular region [GO:0005576],GO:0005576,cellular_component,"('GO:0110165',)","('GO:0043083', 'GO:0099544', 'GO:0098595', 'GO...",362.0,40891.5038,...,7.0,24.0,16.0,14.0,10.0,27.0,22.0,15.0,11.0,20.0
2,O06724,MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...,"['cytoplasm [GO:0005737]', 'acetylpyruvate hyd...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",301.0,33145.5860,...,9.0,7.0,16.0,8.0,10.0,22.0,19.0,17.0,1.0,6.0


In [42]:
# ── 2.2  Quality checks ────────────────────────────────────────────────────
print('Null counts (columns with nulls only):')
null_counts = df2_raw.isnull().sum()
print(null_counts[null_counts > 0])
print()

# Drop rows where physicochemical features or GO label are missing
feature_cols2 = ['seq_length', 'mol_weight', 'pI', 'gravy', 'instability',
                 'aromaticity', 'helix', 'turn', 'sheet']
aa_cols2 = [f'aa_{a}' for a in AA]

df2_raw = df2_raw.dropna(subset=feature_cols2 + aa_cols2 + ['GO_id']).reset_index(drop=True)
print(f'Rows after dropping nulls: {len(df2_raw)}')

Null counts (columns with nulls only):
seq_length     15795
mol_weight     15795
pI             15795
gravy          15795
instability    15795
aromaticity    15795
helix          15795
turn           15795
sheet          15795
aa_A           15795
aa_C           15795
aa_D           15795
aa_E           15795
aa_F           15795
aa_G           15795
aa_H           15795
aa_I           15795
aa_K           15795
aa_L           15795
aa_M           15795
aa_N           15795
aa_P           15795
aa_Q           15795
aa_R           15795
aa_S           15795
aa_T           15795
aa_V           15795
aa_W           15795
aa_Y           15795
dtype: int64

Rows after dropping nulls: 65205


The dataset has one row per (protein, GO term) pair. We collapse it to one row per protein by keeping the most frequent GO term as the label — matching the approach used in the notebook.

In [43]:
# ── 2.3  Do not collapse to one row per protein (retain all GO annotations) ───────────────────
# Instead of collapsing, we will use the df2_raw after null dropping.
# This means each row will represent a (protein, GO term) pair.
df2 = df2_raw.copy()
df2 = df2.rename(columns={'Entry': 'EntryID', 'GO_id': 'GO_label'})

print(f'Shape after retaining all GO terms: {df2.shape}')
print(f'Unique GO labels                 : {df2["GO_label"].nunique()}')
df2.head()

Shape after retaining all GO terms: (65205, 37)
Unique GO labels                 : 20


,EntryID,Sequence,GO_list,GO,GO_label,GO_namespace,GO_parents,GO_children,seq_length,mol_weight,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,C0SPC1,MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...,"['cytoplasm [GO:0005737]', 'ATP binding [GO:00...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",269.0,30790.1862,...,12.0,11.0,11.0,10.0,9.0,14.0,8.0,14.0,10.0,8.0
1,O05512,MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...,"['extracellular region [GO:0005576]', 'mannan ...",extracellular region [GO:0005576],GO:0005576,cellular_component,"('GO:0110165',)","('GO:0043083', 'GO:0099544', 'GO:0098595', 'GO...",362.0,40891.5038,...,7.0,24.0,16.0,14.0,10.0,27.0,22.0,15.0,11.0,20.0
2,O06724,MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...,"['cytoplasm [GO:0005737]', 'acetylpyruvate hyd...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",301.0,33145.5860,...,9.0,7.0,16.0,8.0,10.0,22.0,19.0,17.0,1.0,6.0
3,O08394,MKETSPIPQPKTFGPLGNLPLIDKDKPTLSLIKLAEEQGPIFQIHT...,"['cytosol [GO:0005829]', 'aromatase activity [...",cytosol [GO:0005829],GO:0005829,cellular_component,"('GO:0110165',)","('GO:0099522',)",1061.0,119466.8051,...,23.0,30.0,56.0,50.0,71.0,59.0,58.0,62.0,12.0,32.0
4,O31616,MKRHYEAVVIGGGIIGSAIAYYLAKENKNTALFESGTMGGRTTSAA...,"['cytoplasm [GO:0005737]', 'FAD binding [GO:00...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",369.0,40936.3550,...,13.0,11.0,14.0,9.0,16.0,20.0,13.0,26.0,7.0,12.0


In [44]:
# ── 2.4  Normalise amino-acid counts → frequencies ────────────────────────
# The raw aa_* columns are raw counts; divide by seq_length to get fractions
# (consistent with Dataset 1 and Dataset 3 feature engineering)
aa_sum = df2[aa_cols2].sum(axis=1)
# Only normalise if values look like counts (max > 1 across the column)
if df2[aa_cols2].max().max() > 1.0:
    df2[aa_cols2] = df2[aa_cols2].div(aa_sum, axis=0)
    print('✔ Amino-acid columns normalised to frequencies')
else:
    print('✔ Amino-acid columns already in frequency form')

print(df2[aa_cols2].describe().round(4).T[['mean','std','min','max']])

✔ Amino-acid columns normalised to frequencies
        mean     std     min     max
aa_A  0.0948  0.0275  0.0427  0.2314
aa_C  0.0095  0.0086  0.0000  0.0578
aa_D  0.0551  0.0194  0.0140  0.1046
aa_E  0.0656  0.0242  0.0061  0.1322
aa_F  0.0377  0.0176  0.0000  0.1006
aa_G  0.0740  0.0230  0.0132  0.1480
aa_H  0.0229  0.0139  0.0000  0.1020
aa_I  0.0635  0.0202  0.0153  0.1221
aa_K  0.0575  0.0231  0.0000  0.1273
aa_L  0.0986  0.0265  0.0233  0.2000
aa_M  0.0295  0.0105  0.0067  0.0653
aa_N  0.0385  0.0137  0.0083  0.0926
aa_P  0.0422  0.0173  0.0063  0.1674
aa_Q  0.0393  0.0162  0.0083  0.0878
aa_R  0.0537  0.0251  0.0000  0.1600
aa_S  0.0536  0.0161  0.0000  0.1164
aa_T  0.0525  0.0155  0.0127  0.1007
aa_V  0.0722  0.0213  0.0121  0.1342
aa_W  0.0119  0.0101  0.0000  0.0474
aa_Y  0.0275  0.0129  0.0000  0.0800


---
# Dataset 3 : Human Protein Sequences & Function Annotations

This dataset ships as two separate files: a FASTA file of sequences and a tab-separated annotation file. We parse both then merge them on protein ID.

In [19]:
# ── 3.1  Parse FASTA sequences ─────────────────────────────────────────────
fasta_records = []

# The kagglehub.dataset_download function extracts the contents directly into the specified directory.
# So, we should read the files directly from the directory 'data/raw/human_protein'.
fasta_file_path = RAW_DIR / 'human_protein' / 'ALL-HUMAN-0001 SEQUENCES.fasta'
annot_file_path = RAW_DIR / 'human_protein' / 'ALL-HUMAN-0001-ANNOTATIONS.txt'

print(f'FASTA file : {fasta_file_path}')
print(f'Annot file : {annot_file_path}')

# Open the FASTA file directly
with open(fasta_file_path, 'r') as f:
    cur_id, cur_seq = None, []
    for line in f:
        line = line.strip()
        if line.startswith('>'):
            if cur_id:
                fasta_records.append((cur_id, ''.join(cur_seq)))
            # UniProt header format: >sp|ACCESSION|NAME ...
            parts = line[1:].split('|')
            cur_id = parts[1] if len(parts) >= 2 else line[1:].split()[0]
            cur_seq = []
        else:
            cur_seq.append(line)
    if cur_id:
        fasta_records.append((cur_id, ''.join(cur_seq)))

df3_seq = pd.DataFrame(fasta_records, columns=['Entry', 'Sequence'])
print(f'\nFASTA sequences parsed: {len(df3_seq)}')
df3_seq.head()

FASTA file : data/raw/human_protein/ALL-HUMAN-0001 SEQUENCES.fasta
Annot file : data/raw/human_protein/ALL-HUMAN-0001-ANNOTATIONS.txt

FASTA sequences parsed: 70956


,Entry,Sequence
0,P27361,MAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQ...
1,P53779,MSLHFLYYCSEPTLDVKIAFCQGFDKQVDVSYIAKHYNMSKSKVDN...
2,Q15049,MTQEPFREELAYDRMPTLERGRQDPASYAPDAKPSDLQLSKRLPPC...
3,Q9UHC1,MIKCLSVEVQAKLRSGLAISSLGQCVEELALNSIDAEAKCVAVRVN...
4,P0DMT0,MTGKNWILISTTTPKSLEDEIVGRLLKILFVIFVDLISIIYVVITS


In [21]:
# ── 3.2  Parse GO annotation file ──────────────────────────────────────────
annot_records = []

# The annotation file was already extracted to the raw data directory
# and its path is available in `annot_file_path` from the previous cell.
with open(annot_file_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Format: PROTEIN_ID  GO:XXXXXXX;  F:description  EVIDENCE:SOURCE
        parts = line.split()
        if len(parts) < 2:
            continue
        protein_id = parts[0]
        go_raw     = parts[1]
        # Strip trailing semicolon from GO term
        go_term = go_raw.rstrip(';')
        if go_term.startswith('GO:'):
            annot_records.append({'Entry': protein_id, 'GO_id': go_term})

df3_annot = pd.DataFrame(annot_records)
print(f'Annotation rows parsed: {len(df3_annot)}')
print(f'Unique proteins       : {df3_annot["Entry"].nunique()}')
df3_annot.head()

Annotation rows parsed: 68157
Unique proteins       : 30608


,Entry,GO_id
0,P27361,GO:0005524
1,P27361,GO:0016301
2,P27361,GO:0004707
3,P27361,GO:0019902
4,P27361,GO:0004674


We collapse annotations to one GO label per protein (most frequent), then merge with the sequences.

In [28]:
# ── 3.3  Collapse to one GO label per protein ──────────────────────────────
df3_labels = (
    df3_annot
    .groupby('Entry')['GO_id']
    .apply(list)
    .reset_index()
)
df3_labels['GO_id'] = df3_labels['GO_id'].apply(pick_label)

# ── 3.4  Merge sequences + labels ─────────────────────────────────────────
df3 = pd.merge(df3_seq, df3_labels, on='Entry', how='inner')
print(f'Shape after merge: {df3.shape}')
df3.head()

Shape after merge: (30608, 3)


,Entry,Sequence,GO_id
0,P27361,MAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQ...,GO:0005524
1,P53779,MSLHFLYYCSEPTLDVKIAFCQGFDKQVDVSYIAKHYNMSKSKVDN...,GO:0005524
2,Q15049,MTQEPFREELAYDRMPTLERGRQDPASYAPDAKPSDLQLSKRLPPC...,GO:0032403
3,Q9UHC1,MIKCLSVEVQAKLRSGLAISSLGQCVEELALNSIDAEAKCVAVRVN...,GO:0005524
4,P0DMT0,MTGKNWILISTTTPKSLEDEIVGRLLKILFVIFVDLISIIYVVITS,GO:0004857


In [29]:
# ── 3.5  Quality checks ────────────────────────────────────────────────────
print('Null values:')
print(df3.isnull().sum())

df3 = df3.dropna(subset=['Sequence', 'GO_id']).reset_index(drop=True)

# Remove sequences with non-standard amino acid characters
valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
mask3 = df3['Sequence'].apply(lambda s: set(str(s).upper()).issubset(valid_aa))
removed3 = (~mask3).sum()
df3 = df3[mask3].reset_index(drop=True)
print(f'Rows removed (invalid AA): {removed3}')
print(f'Remaining rows           : {len(df3)}')
print(f'Unique GO labels         : {df3["GO_id"].nunique()}')

Null values:
Entry       0
Sequence    0
GO_id       0
dtype: int64
Rows removed (invalid AA): 2042
Remaining rows           : 28566
Unique GO labels         : 2062


Dataset 3 sequences have no pre-computed physicochemical features, so we engineer them from scratch using BioPython — matching the feature schema of the other two datasets.

In [26]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.9 MB/s eta 0:00:00


In [30]:
# ── 3.6  Engineer physicochemical features from sequence ──────────────────
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    BIOPYTHON = True
except ImportError:
    BIOPYTHON = False
    print('BioPython not available – computing features manually')

def compute_features(seq):
    """Return a dict of physicochemical features for one sequence."""
    seq = str(seq).upper()
    L   = len(seq)
    c   = Counter(seq)

    # Amino-acid composition (fractions)
    aa_comp = {f'aa_{a}': c.get(a, 0) / L for a in AA}

    if BIOPYTHON:
        # Replace ambiguous characters for BioPython
        clean = re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', 'A', seq)
        pa = ProteinAnalysis(clean)
        phys = {
            'seq_length' : L,
            'mol_weight' : pa.molecular_weight(),
            'pI'         : pa.isoelectric_point(),
            'gravy'      : pa.gravy(),
            'instability': pa.instability_index(),
            'aromaticity': pa.aromaticity(),
            'helix'      : pa.secondary_structure_fraction()[0],
            'turn'       : pa.secondary_structure_fraction()[1],
            'sheet'      : pa.secondary_structure_fraction()[2],
        }
    else:
        # Minimal fallback: seq_length only
        phys = {'seq_length': L, 'mol_weight': np.nan, 'pI': np.nan,
                'gravy': np.nan, 'instability': np.nan, 'aromaticity': np.nan,
                'helix': np.nan, 'turn': np.nan, 'sheet': np.nan}

    return {**phys, **aa_comp}

print('Computing features for Dataset 3… (this may take a minute)')
feature_rows = df3['Sequence'].apply(compute_features)
df3_feats    = pd.DataFrame(list(feature_rows))
df3          = pd.concat([df3.reset_index(drop=True), df3_feats], axis=1)

print(f'✔ Features computed – shape: {df3.shape}')
df3.head(3)

Computing features for Dataset 3… (this may take a minute)
✔ Features computed – shape: (28566, 32)


,Entry,Sequence,GO_id,seq_length,mol_weight,pI,gravy,instability,aromaticity,helix,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,P27361,MAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQ...,GO:0005524,379,43135.0670,6.280447,-0.313984,43.062797,0.089710,0.332454,...,0.026385,0.034301,0.058047,0.044855,0.060686,0.047493,0.050132,0.047493,0.007916,0.047493
1,P53779,MSLHFLYYCSEPTLDVKIAFCQGFDKQVDVSYIAKHYNMSKSKVDN...,GO:0005524,464,52584.8692,6.329727,-0.298276,42.812737,0.088362,0.316810,...,0.036638,0.043103,0.056034,0.045259,0.038793,0.079741,0.040948,0.077586,0.008621,0.047414
2,Q15049,MTQEPFREELAYDRMPTLERGRQDPASYAPDAKPSDLQLSKRLPPC...,GO:0032403,377,41140.7810,7.458663,0.435013,46.004244,0.087533,0.339523,...,0.023873,0.031830,0.058355,0.029178,0.042440,0.100796,0.042440,0.087533,0.013263,0.021220


In [31]:
# ── 3.7  Final cleanup – drop rows where feature computation failed ────────
num_cols3 = ['seq_length', 'mol_weight', 'pI', 'gravy', 'instability',
             'aromaticity', 'helix', 'turn', 'sheet']
df3 = df3.dropna(subset=num_cols3).reset_index(drop=True)
print(f'Shape after final cleanup: {df3.shape}')

Shape after final cleanup: (28566, 32)


---
# Label Encoding
String class labels are encoded to integers so they are compatible with all three model families (LR, SVM, CNN).

In [46]:
# Dataset 1 – 5-class functional category
le1 = LabelEncoder()
df1['Class_enc'] = le1.fit_transform(df1['Class'])
print('Dataset 1 label mapping:')
for i, c in enumerate(le1.classes_):
    print(f'  {i} → {c}')

# Dataset 2 – GO term label
le2 = LabelEncoder()
df2['GO_label_enc'] = le2.fit_transform(df2['GO_label'])
print(f'\nDataset 2 unique GO labels : {len(le2.classes_)}')

# Dataset 3 – GO term label
le3 = LabelEncoder()
df3['GO_id_enc'] = le3.fit_transform(df3['GO_id'])
print(f'Dataset 3 unique GO labels : {len(le3.classes_)}')

Dataset 1 label mapping:
  0 → Enzyme
  1 → Others
  2 → Receptor
  3 → Structural
  4 → Transporter

Dataset 2 unique GO labels : 20
Dataset 3 unique GO labels : 2062


---
# Feature Engineering
A single unified feature representation was constructed for each dataset by combining physicochemical properties and amino acid composition, ensuring that the same input space could be consistently used across Logistic Regression, SVM, and CNN models for fair performance comparison.

In [33]:
# ── Helper: amino-acid composition from raw sequence ──────────────────────
def aa_comp(seq):
    c = Counter(seq)
    L = len(seq)
    return np.array([c.get(a, 0) / L for a in AA])

In [34]:
# ── Dataset 1: physicochemical + AA composition ───────────────────────────
num1 = ['Molecular_Weight', 'Isoelectric_Point', 'Hydrophobicity',
        'Net_Charge', 'Polar_Ratio', 'NonPolar_Ratio', 'Sequence_Length']

X1_num = df1[num1].values
X1_seq = np.vstack(df1['Sequence'].apply(aa_comp))
X1     = np.hstack([X1_num, X1_seq])
y1     = df1['Class_enc'].values

print(f'X1 shape: {X1.shape}  |  y1 shape: {y1.shape}')

X1 shape: (16000, 27)  |  y1 shape: (16000,)


In [47]:
# ── Dataset 2: physicochemical + AA composition (already computed) ─────────
num2 = ['seq_length', 'mol_weight', 'pI', 'gravy', 'instability',
        'aromaticity', 'helix', 'turn', 'sheet']
aa_cols2 = [f'aa_{a}' for a in AA]

X2 = df2[num2 + aa_cols2].values
y2 = df2['GO_label_enc'].values

print(f'X2 shape: {X2.shape}  |  y2 shape: {y2.shape}')

X2 shape: (65205, 29)  |  y2 shape: (65205,)


In [36]:
# ── Dataset 3: physicochemical + AA composition ───────────────────────────
num3    = ['seq_length', 'mol_weight', 'pI', 'gravy', 'instability',
           'aromaticity', 'helix', 'turn', 'sheet']
aa_cols3 = [f'aa_{a}' for a in AA]

X3 = df3[num3 + aa_cols3].values
y3 = df3['GO_id_enc'].values

print(f'X3 shape: {X3.shape}  |  y3 shape: {y3.shape}')

X3 shape: (28566, 29)  |  y3 shape: (28566,)


---
# CNN Reshaping
A Conv1D layer expects 3D input, because it is designed to slide a filter over a sequence-like structure.

The feature vectors were reshaped into a 3D tensor format to enable 1D convolution over the feature axis, allowing the model to learn local interactions between adjacent biochemical descriptors.

In [48]:
X1_cnn = X1.reshape(X1.shape[0], X1.shape[1], 1)
X2_cnn = X2.reshape(X2.shape[0], X2.shape[1], 1)
X3_cnn = X3.reshape(X3.shape[0], X3.shape[1], 1)

print('Flat shapes  :', X1.shape, X2.shape, X3.shape)
print('CNN shapes   :', X1_cnn.shape, X2_cnn.shape, X3_cnn.shape)

Flat shapes  : (16000, 27) (65205, 29) (28566, 29)
CNN shapes   : (16000, 27, 1) (65205, 29, 1) (28566, 29, 1)


---
# Save Processed Data
All arrays and cleaned DataFrames are saved so downstream modelling notebooks can load them directly without re-running this pipeline.

In [49]:
# ── Save numpy arrays (flat + CNN) ────────────────────────────────────────
np.save(PROC_DIR / 'X1.npy',     X1)
np.save(PROC_DIR / 'y1.npy',     y1)
np.save(PROC_DIR / 'X1_cnn.npy', X1_cnn)

np.save(PROC_DIR / 'X2.npy',     X2)
np.save(PROC_DIR / 'y2.npy',     y2)
np.save(PROC_DIR / 'X2_cnn.npy', X2_cnn)

np.save(PROC_DIR / 'X3.npy',     X3)
np.save(PROC_DIR / 'y3.npy',     y3)
np.save(PROC_DIR / 'X3_cnn.npy', X3_cnn)

# ── Save cleaned DataFrames ───────────────────────────────────────────────
df1.to_csv(PROC_DIR / 'df1_cleaned.csv', index=False)
df2.to_csv(PROC_DIR / 'df2_cleaned.csv', index=False)
df3.to_csv(PROC_DIR / 'df3_cleaned.csv', index=False)

print('✔ All processed files saved to', str(PROC_DIR))
print()
for f in sorted(PROC_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<25}  {size_kb:>8.1f} KB')

✔ All processed files saved to data/processed

  X1.npy                       4039.5 KB
  X1_cnn.npy                   4039.5 KB
  X2.npy                      14773.1 KB
  X2_cnn.npy                  14773.1 KB
  X3.npy                       6472.1 KB
  X3_cnn.npy                   6472.1 KB
  df1_cleaned.csv              4892.1 KB
  df2_cleaned.csv             92682.7 KB
  df3_cleaned.csv             28639.6 KB
  y1.npy                        125.1 KB
  y2.npy                        509.5 KB
  y3.npy                        223.3 KB


In [50]:
summary = pd.DataFrame({
    'Dataset'     : ['Dataset 1 (Simulated)',
                     'Dataset 2 (GO Annotations)',
                     'Dataset 3 (Human Proteins)'],
    'Samples'     : [X1.shape[0], X2.shape[0], X3.shape[0]],
    'Features'    : [X1.shape[1], X2.shape[1], X3.shape[1]],
    'CNN shape'   : [str(X1_cnn.shape), str(X2_cnn.shape), str(X3_cnn.shape)],
    'Unique labels': [len(np.unique(y1)), len(np.unique(y2)), len(np.unique(y3))],
    'Target column': ['Class (5-class)', 'GO_label (GO term)', 'GO_id (GO term)'],
})

summary

,Dataset,Samples,Features,CNN shape,Unique labels,Target column
0,Dataset 1 (Simulated),16000,27,"(16000, 27, 1)",5,Class (5-class)
1,Dataset 2 (GO Annotations),65205,29,"(65205, 29, 1)",20,GO_label (GO term)
2,Dataset 3 (Human Proteins),28566,29,"(28566, 29, 1)",2062,GO_id (GO term)
